# LeetCode 3710 — Maximum Partition Factor bằng Python

- Link: https://leetcode.com/problems/maximum-partition-factor/
- Độ khó: **Hard**
- Chủ đề: Tìm nhị phân trên đáp án, **đồ thị hai phía** (bipartite), BFS tô màu, **DSU có chẵn lẻ** (Kruskal ngược), khoảng cách Manhattan

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxPartitionFactor(self, points: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Chuyển "tối đa hoá khoảng cách nhỏ nhất trong cùng nhóm" thành câu hỏi quyết định: **có chia được hai nhóm sao cho mọi cặp cùng nhóm cách nhau `≥ d`?**
2. Nhận ra câu hỏi đó ⇔ đồ thị "cạnh nối các cặp có khoảng cách `< d`" là **hai phía**.
3. Tính đơn điệu theo `d` ⇒ tìm nhị phân trên tập khoảng cách.
4. Cách gọn hơn: duyệt cạnh theo khoảng cách tăng dần, **DSU có chẵn lẻ** phát hiện chu trình lẻ đầu tiên — trọng số của cạnh đó là đáp án.
5. Xử lý trường hợp đặc biệt `n = 2`.

## 2. Đề bài & ví dụ

Chia `n` điểm thành **đúng hai nhóm khác rỗng**. Hệ số của cách chia = khoảng cách Manhattan **nhỏ nhất** giữa hai điểm **cùng nhóm**. Tìm hệ số lớn nhất. Khi `n = 2` (hai nhóm đơn), quy ước hệ số là `0`.

**Ví dụ 1:** `[[0,0],[0,2],[2,0],[2,2]]` → `4`.

**Ví dụ 2:** `[[0,0],[0,1],[10,0]]` → `11`.

**Ràng buộc:** `2 ≤ n ≤ 500`, toạ độ `|·| ≤ 10^8`.

In [1]:
from typing import List
from collections import deque
from itertools import combinations
import random, time

examples = [([[0, 0], [0, 2], [2, 0], [2, 2]], 4), ([[0, 0], [0, 1], [10, 0]], 11)]
for pts, exp in examples:
    print(pts, "-> kỳ vọng", exp)

[[0, 0], [0, 2], [2, 0], [2, 2]] -> kỳ vọng 4
[[0, 0], [0, 1], [10, 0]] -> kỳ vọng 11


## 3. Kiến thức nền

### 3.1. Khoảng cách Manhattan và danh sách cạnh

`dist(i, j) = |x_i − x_j| + |y_i − y_j|`. Với `n = 500` có `n(n−1)/2 ≈ 1,25·10^5` cặp — liệt kê và sắp xếp được.

In [2]:
def dist(p, q):
    return abs(p[0] - q[0]) + abs(p[1] - q[1])

pts = examples[0][0]
print(sorted((dist(pts[i], pts[j]), i, j) for i, j in combinations(range(len(pts)), 2)))

[(2, 0, 1), (2, 0, 2), (2, 1, 3), (2, 2, 3), (4, 0, 3), (4, 1, 2)]


### 3.2. Đồ thị hai phía

Đồ thị chia được thành hai tập sao cho mọi cạnh nối hai tập khác nhau ⇔ **không có chu trình lẻ**. Kiểm tra bằng BFS tô hai màu.

In [3]:
def is_bipartite(n, adj):
    color = [-1] * n
    for s in range(n):
        if color[s] != -1: continue
        color[s] = 0; q = deque([s])
        while q:
            u = q.popleft()
            for v in adj[u]:
                if color[v] == -1: color[v] = color[u] ^ 1; q.append(v)
                elif color[v] == color[u]: return False
    return True

print(is_bipartite(4, [[1, 3], [0, 2], [1, 3], [2, 0]]), is_bipartite(3, [[1, 2], [0, 2], [0, 1]]))

True False


### 3.3. DSU có chẵn lẻ

Mỗi phần tử lưu `parity` so với cha. `find(x)` trả về `(gốc, chẵn lẻ của x so với gốc)`. Thêm ràng buộc "x và y khác màu": nếu cùng gốc mà cùng chẵn lẻ ⇒ **mâu thuẫn** (chu trình lẻ); khác gốc ⇒ nối hai cây với độ lệch phù hợp.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hệ số `≥ d` ⇔ mọi cặp có khoảng cách `< d` phải nằm **khác nhóm** ⇔ đồ thị `G_d` (cạnh: cặp có khoảng cách `< d`) là hai phía. Tăng `d` chỉ thêm cạnh ⇒ tính chất "hai phía" chỉ có thể mất đi ⇒ **đơn điệu**.

Đáp án là `d` lớn nhất còn hai phía. Nếu duyệt cạnh theo khoảng cách tăng dần, `G` mất tính hai phía ngay khi thêm cạnh `e` có trọng số `w` tạo chu trình lẻ; khi đó `d = w` vẫn được (chỉ cạnh `< w` bị cấm) còn `d > w` thì không ⇒ **đáp án = `w`**.

Ví dụ 2: cạnh `1 (0–1)`, `10 (0–2)`, `11 (1–2)`: cạnh thứ ba tạo tam giác (chu trình lẻ) ⇒ `11`. ✔

## 5. Cách 1 — Brute Force: thử mọi cách chia

In [4]:
def brute(points):
    n = len(points)
    if n == 2: return 0
    best = 0
    for mask in range(1, (1 << n) - 1):
        groups = [[i for i in range(n) if mask >> i & 1], [i for i in range(n) if not mask >> i & 1]]
        f = min((dist(points[i], points[j]) for g in groups for i, j in combinations(g, 2)), default=None)
        if f is not None: best = max(best, f)
    return best

for pts, exp in examples:
    print(brute(pts), "kỳ vọng", exp)

4 kỳ vọng 4
11 kỳ vọng 11


**Nút thắt:** `O(2^n · n²)` — chỉ cho `n ≤ 12`.

## 6. Ý tưởng thuật toán / bước đột phá

Đáp án luôn là **một khoảng cách** giữa hai điểm (cặp cùng nhóm gần nhất), trừ `n = 2`. Hai cách khai thác tính đơn điệu:

- **Tìm nhị phân** trên danh sách khoảng cách đã sắp: kiểm tra `G_d` hai phía bằng BFS `O(n²)` ⇒ `O(n² log n)`.
- **Kruskal + DSU chẵn lẻ:** duyệt cạnh tăng dần, gặp cạnh mâu thuẫn đầu tiên ⇒ trả trọng số ⇒ `O(n² log n)` (do sắp xếp), không cần BFS lặp lại.

Với `n ≥ 3`, đồ thị đầy đủ có tam giác ⇒ chắc chắn có cạnh mâu thuẫn. Hai nhóm luôn khác rỗng được: nếu tô màu đặt mọi đỉnh một bên (khi không có cạnh), chuyển một đỉnh sang bên kia.

## 7. Các cách tiếp theo

- **Cách 2:** tìm nhị phân trên khoảng cách + BFS tô màu.
- **Cách 3:** DSU có chẵn lẻ, duyệt cạnh tăng dần (lời giải nộp).
- **Cách 4:** tìm nhị phân trên **giá trị nguyên** `d ∈ [0, 4·10^8]` (không cần danh sách khoảng cách phân biệt) — minh hoạ "binary search on answer" thuần tuý.

### 7.1. Cách 2 — Tìm nhị phân trên danh sách khoảng cách

In [5]:
def feasible(points, d):
    n = len(points)
    adj = [[j for j in range(n) if j != i and dist(points[i], points[j]) < d] for i in range(n)]
    return is_bipartite(n, adj)

def binary_search_dists(points):
    n = len(points)
    if n == 2: return 0
    ds = sorted({dist(points[i], points[j]) for i, j in combinations(range(n), 2)})
    lo, hi = 0, len(ds) - 1                         # feasible(ds[0]) luôn đúng (không cạnh nào < min)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if feasible(points, ds[mid]): lo = mid
        else: hi = mid - 1
    return ds[lo]

for pts, exp in examples:
    print(binary_search_dists(pts), "kỳ vọng", exp)

4 kỳ vọng 4
11 kỳ vọng 11


### 7.2. Cách 3 — DSU có chẵn lẻ (lời giải chính)

In [6]:
def dsu_parity(points):
    n = len(points)
    if n == 2: return 0
    edges = sorted((dist(points[i], points[j]), i, j) for i in range(n) for j in range(i + 1, n))
    parent = list(range(n)); par = [0] * n            # par[x] = chẵn lẻ của x so với parent[x]

    def find(x):
        path = []
        while parent[x] != x:
            path.append(x); x = parent[x]
        root = x; acc = 0
        for y in reversed(path):                     # nén đường, tính lại chẵn lẻ so với gốc
            acc ^= par[y]; par[y] = acc; parent[y] = root
        return root

    for w, i, j in edges:
        ri, rj = find(i), find(j)
        pi = par[i] if i != ri else 0; pj = par[j] if j != rj else 0
        if ri == rj:
            if pi == pj: return w                     # i, j buộc cùng màu nhưng cần khác màu
        else:
            parent[ri] = rj; par[ri] = pi ^ pj ^ 1
    return 0

for pts, exp in examples:
    print(dsu_parity(pts), "kỳ vọng", exp)

4 kỳ vọng 4
11 kỳ vọng 11


### 7.3. Cách 4 — Tìm nhị phân trên giá trị nguyên

In [7]:
def binary_search_values(points):
    if len(points) == 2: return 0
    lo, hi = 0, 4 * 10**8 + 1                         # feasible(lo) đúng, tìm d lớn nhất
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if feasible(points, mid): lo = mid
        else: hi = mid - 1
    return lo

for pts, exp in examples:
    print(binary_search_values(pts), "kỳ vọng", exp)

4 kỳ vọng 4
11 kỳ vọng 11


**Nhận xét:** kết quả luôn rơi đúng vào một khoảng cách (vì `feasible` chỉ đổi giá trị tại các khoảng cách), nhưng cần ~29 lần BFS thay vì ~17.

## 8. TRACE từng bước

In [8]:
def trace(points):
    n = len(points)
    edges = sorted((dist(points[i], points[j]), i, j) for i in range(n) for j in range(i + 1, n))
    color_parent = list(range(n)); par = [0] * n
    def find(x):
        p = 0
        while color_parent[x] != x: p ^= par[x]; x = color_parent[x]
        return x, p
    for w, i, j in edges:
        (ri, pi), (rj, pj) = find(i), find(j)
        if ri == rj:
            status = "MÂU THUẪN -> đáp án" if pi == pj else "đã nhất quán"
            print(f"cạnh {w:>3} ({i},{j}): cùng thành phần, chẵn lẻ {pi},{pj} -> {status}")
            if pi == pj: return w
        else:
            color_parent[ri] = rj; par[ri] = pi ^ pj ^ 1
            print(f"cạnh {w:>3} ({i},{j}): nối hai thành phần, buộc khác màu")

trace(examples[0][0])

cạnh   2 (0,1): nối hai thành phần, buộc khác màu
cạnh   2 (0,2): nối hai thành phần, buộc khác màu
cạnh   2 (1,3): nối hai thành phần, buộc khác màu
cạnh   2 (2,3): cùng thành phần, chẵn lẻ 1,0 -> đã nhất quán
cạnh   4 (0,3): cùng thành phần, chẵn lẻ 0,0 -> MÂU THUẪN -> đáp án


4

In [9]:
pts = [[0, 0], [3, 0], [6, 0], [0, 4], [9, 9]]
print("đáp án:", trace(pts), "| brute:", brute(pts))

cạnh   3 (0,1): nối hai thành phần, buộc khác màu
cạnh   3 (1,2): nối hai thành phần, buộc khác màu
cạnh   4 (0,3): nối hai thành phần, buộc khác màu
cạnh   6 (0,2): cùng thành phần, chẵn lẻ 1,1 -> MÂU THUẪN -> đáp án
đáp án: 6 | brute: 6


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def maxPartitionFactor(self, points: List[List[int]]) -> int:
        n = len(points)
        if n == 2:
            return 0
        edges = []
        for i in range(n):
            xi, yi = points[i]
            for j in range(i + 1, n):
                edges.append((abs(xi - points[j][0]) + abs(yi - points[j][1]), i, j))
        edges.sort()
        parent = list(range(n))
        par = [0] * n

        def find(x):
            path = []
            while parent[x] != x:
                path.append(x)
                x = parent[x]
            root, acc = x, 0
            for y in reversed(path):
                acc ^= par[y]
                par[y] = acc
                parent[y] = root
            return root

        for w, i, j in edges:
            ri, rj = find(i), find(j)
            pi = par[i] if i != ri else 0
            pj = par[j] if j != rj else 0
            if ri == rj:
                if pi == pj:
                    return w
            else:
                parent[ri] = rj
                par[ri] = pi ^ pj ^ 1
        return 0

sol = Solution()
for pts, exp in examples:
    got = sol.maxPartitionFactor(pts)
    print(pts, "->", got, "✔" if got == exp else "✘")

[[0, 0], [0, 2], [2, 0], [2, 2]] -> 4 ✔
[[0, 0], [0, 1], [10, 0]] -> 11 ✔


## 10. Vì sao thuật toán đúng

1. **Quyết định:** hệ số `≥ d` ⇔ tồn tại 2-tô màu (hai nhóm) mà mọi cặp khoảng cách `< d` khác màu ⇔ `G_d` hai phía (và hai nhóm khác rỗng — luôn sửa được khi `n ≥ 3` như mục 6).
2. **Đơn điệu:** `d₁ < d₂` ⇒ `G_{d₁} ⊆ G_{d₂}` ⇒ nếu `G_{d₂}` hai phía thì `G_{d₁}` cũng vậy.
3. **DSU chẵn lẻ:** sau khi xử lý các cạnh trọng số `< w`, các ràng buộc "khác màu" nhất quán ⇔ đồ thị con hai phía; DSU lưu chính xác quan hệ chẵn lẻ giữa các đỉnh cùng thành phần (nén đường giữ chẵn lẻ bằng XOR dọc đường).
4. **Đáp án:** gọi `e` là cạnh đầu tiên (theo thứ tự tăng) gây mâu thuẫn, trọng số `w`. Mọi cạnh `< w` đều đã xử lý mà không mâu thuẫn ⇒ `G_w` hai phía ⇒ hệ số `≥ w` khả thi. `G_{w+1}` chứa mọi cạnh `≤ w` đã xử lý **tới `e`** ⇒ có chu trình lẻ ⇒ không khả thi. Các cạnh cùng trọng số `w` xử lý trước `e` không làm thay đổi kết luận. Vậy đáp án = `w`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi cách chia | `O(2^n n²)` | `O(n)` |
| 2. Nhị phân trên khoảng cách + BFS | `O(n² log n)` | `O(n²)` |
| 3. DSU chẵn lẻ | `O(n² log n)` (sắp xếp) | `O(n²)` |
| 4. Nhị phân trên giá trị + BFS | `O(n² log C)` | `O(n²)` |

## 12. Bộ test

In [11]:
METHODS = [brute, binary_search_dists, dsu_parity, binary_search_values, lambda p: Solution().maxPartitionFactor(p)]
for pts, exp in examples:
    assert all(f(pts) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [12]:
# 12.2 ca biên
assert Solution().maxPartitionFactor([[0, 0], [5, 5]]) == 0                       # n = 2
assert Solution().maxPartitionFactor([[0, 0], [0, 0], [0, 0]]) == 0               # điểm trùng
assert Solution().maxPartitionFactor([[0, 0], [0, 0], [7, 0]]) == 7
assert Solution().maxPartitionFactor([[-10**8, -10**8], [10**8, 10**8], [0, 0]]) == 4 * 10**8
print("Ca biên: OK")

Ca biên: OK


In [13]:
# 12.3 ngẫu nhiên
rng = random.Random(3710)
for _ in range(400):
    n = rng.randint(2, 9)
    pts = [[rng.randint(-5, 5), rng.randint(-5, 5)] for _ in range(n)]
    b = brute(pts)
    assert all(f(pts) == b for f in METHODS[1:]), pts
print("400 test ngẫu nhiên: OK")

400 test ngẫu nhiên: OK


In [14]:
# 12.4 test lớn n = 500
rng = random.Random(1)
pts = [[rng.randint(-10**8, 10**8), rng.randint(-10**8, 10**8)] for _ in range(500)]
t0 = time.time(); x = Solution().maxPartitionFactor(pts); t1 = time.time()
y = binary_search_dists(pts); t2 = time.time()
print("đáp án:", x, "| DSU:", round(t1 - t0, 2), "s | nhị phân + BFS:", round(t2 - t1, 2), "s | khớp:", x == y)

đáp án: 1834335 | DSU: 0.09 s | nhị phân + BFS: 0.7 s | khớp: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `≤ d` thay vì `< d` khi dựng đồ thị (lệch một cạnh)

In [15]:
def wrong_le(points):
    n = len(points)
    if n == 2: return 0
    ds = sorted({dist(points[i], points[j]) for i, j in combinations(range(n), 2)})
    ok = lambda d: is_bipartite(n, [[j for j in range(n) if j != i and dist(points[i], points[j]) <= d] for i in range(n)])
    return max([d for d in ds if ok(d)], default=0)                     # SAI: <= d

print("sai:", wrong_le(examples[1][0]), "| đúng:", brute(examples[1][0]))

sai: 10 | đúng: 11


### Lỗi 2 — Tham lam: tô màu theo thứ tự cạnh mà **không** dùng DSU (tô cố định khi gặp lần đầu)

In [16]:
def wrong_greedy_color(points):
    n = len(points)
    if n == 2: return 0
    edges = sorted((dist(points[i], points[j]), i, j) for i in range(n) for j in range(i + 1, n))
    color = [-1] * n
    for w, i, j in edges:
        if color[i] == -1 and color[j] == -1: color[i], color[j] = 0, 1
        elif color[i] == -1: color[i] = color[j] ^ 1
        elif color[j] == -1: color[j] = color[i] ^ 1
        elif color[i] == color[j]: return w                              # SAI: màu cố định quá sớm
    return 0

rng = random.Random(9)
while True:
    pts = [[rng.randint(0, 9), rng.randint(0, 9)] for _ in range(rng.randint(3, 7))]
    if wrong_greedy_color(pts) != brute(pts): break
print("phản ví dụ:", pts, "| sai:", wrong_greedy_color(pts), "| đúng:", brute(pts))

phản ví dụ: [[1, 4], [4, 1], [8, 0], [0, 6], [8, 6]] | sai: 6 | đúng: 8


### Lỗi 3 — Quên trường hợp `n = 2`

Không có cặp cùng nhóm ⇒ quy ước `0`; vòng lặp cạnh không gặp mâu thuẫn nên cũng trả `0`, nhưng cách tìm nhị phân có thể trả khoảng cách duy nhất.

In [17]:
def wrong_n2(points):
    n = len(points)
    ds = sorted({dist(points[i], points[j]) for i, j in combinations(range(n), 2)})
    return ds[-1]                                                         # SAI khi n = 2

print("sai:", wrong_n2([[0, 0], [3, 4]]), "| đúng:", Solution().maxPartitionFactor([[0, 0], [3, 4]]))

sai: 7 | đúng: 0


### Lỗi 4 — DSU chẵn lẻ: nén đường mà không cập nhật chẵn lẻ

Nếu `parent[y] = root` nhưng giữ `par[y]` là chẵn lẻ so với **cha cũ**, các truy vấn sau sai.

In [18]:
def wrong_compress(points):
    n = len(points)
    if n == 2: return 0
    edges = sorted((dist(points[i], points[j]), i, j) for i in range(n) for j in range(i + 1, n))
    parent = list(range(n)); par = [0] * n
    def find(x):
        r = x; p = 0
        while parent[r] != r: p ^= par[r]; r = parent[r]
        y = x
        while parent[y] != y:
            nxt = parent[y]; parent[y] = r; y = nxt                        # SAI: không sửa par[y]
        return r, p
    for w, i, j in edges:
        (ri, pi), (rj, pj) = find(i), find(j)
        if ri == rj:
            if pi == pj: return w
        else: parent[ri] = rj; par[ri] = pi ^ pj ^ 1
    return 0

rng = random.Random(21)
while True:
    pts = [[rng.randint(0, 9), rng.randint(0, 9)] for _ in range(rng.randint(4, 9))]
    if wrong_compress(pts) != brute(pts): break
print("phản ví dụ:", pts, "| sai:", wrong_compress(pts), "| đúng:", brute(pts))

phản ví dụ: [[2, 0], [5, 0], [2, 5], [3, 8], [0, 0], [0, 2]] | sai: 5 | đúng: 4


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tối đa hoá giá trị nhỏ nhất" ⇒ tìm nhị phân trên đáp án hoặc duyệt ngưỡng tăng dần.
- "Chia hai nhóm, cặp gần nhau phải khác nhóm" ⇒ **đồ thị hai phía**.
- Thêm cạnh theo thứ tự + kiểm tra chu trình lẻ ⇒ **DSU có chẵn lẻ**.
- `n ≤ 500` ⇒ `O(n² log n)` với mọi cặp điểm là chấp nhận được.

## 15. Biến thể của bài

1. **Chia thành `k` nhóm** (`k ≥ 3`): kiểm tra `k`-tô màu là NP-khó — chỉ làm được với `n` nhỏ.
2. **Khoảng cách Euclid**: cùng lời giải, đổi hàm khoảng cách (so sánh bình phương để tránh số thực).
3. **Bài "chia hai nhóm tù nhân"** (LC 886 *Possible Bipartition*): chỉ cần kiểm tra hai phía.
4. **Tối thiểu hoá khoảng cách lớn nhất giữa hai nhóm**: đối ngẫu, dùng cây khung.

In [19]:
# Biến thể 2: khoảng cách Euclid bình phương
def max_partition_euclid_sq(points):
    n = len(points)
    if n == 2: return 0
    d2 = lambda p, q: (p[0] - q[0]) ** 2 + (p[1] - q[1]) ** 2
    edges = sorted((d2(points[i], points[j]), i, j) for i in range(n) for j in range(i + 1, n))
    parent = list(range(n)); par = [0] * n
    def find(x):
        p = 0
        while parent[x] != x: p ^= par[x]; x = parent[x]
        return x, p
    for w, i, j in edges:
        (ri, pi), (rj, pj) = find(i), find(j)
        if ri == rj:
            if pi == pj: return w
        else: parent[ri] = rj; par[ri] = pi ^ pj ^ 1
    return 0

def brute_euclid(points):
    n = len(points); best = 0
    d2 = lambda p, q: (p[0] - q[0]) ** 2 + (p[1] - q[1]) ** 2
    for mask in range(1, (1 << n) - 1):
        gs = [[i for i in range(n) if mask >> i & 1], [i for i in range(n) if not mask >> i & 1]]
        f = min((d2(points[i], points[j]) for g in gs for i, j in combinations(g, 2)), default=None)
        if f is not None: best = max(best, f)
    return best

rng = random.Random(15)
for _ in range(300):
    pts = [[rng.randint(-4, 4), rng.randint(-4, 4)] for _ in range(rng.randint(3, 8))]
    assert max_partition_euclid_sq(pts) == brute_euclid(pts)
print("Biến thể 2 OK")

Biến thể 2 OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Thử mọi cách chia | Định nghĩa | `O(2^n n²)` | Kiểm chứng |
| 2. Nhị phân + BFS | Đơn điệu + hai phía | `O(n² log n)` | Mẫu tổng quát |
| 3. DSU chẵn lẻ | Cạnh mâu thuẫn đầu tiên | `O(n² log n)` | **Lời giải chuẩn** |
| 4. Nhị phân trên giá trị | Không cần liệt kê ngưỡng | `O(n² log C)` | Khi ngưỡng khó liệt kê |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `[[0,0],[1,0],[2,0]]`.

**Bài 2 (tính tay).** `[[0,0],[4,0],[0,3],[4,3]]`.

**Bài 3 (điền TODO).** Viết `best_split(points)` trả về **một cách chia** đạt hệ số tối ưu (danh sách màu 0/1).

**Bài 4 (tự viết lại).** Cài `possible_bipartition(n, dislikes)` (LC 886) bằng DSU chẵn lẻ.

**Bài 5 (mini challenge).** Đếm số cách chia (không phân biệt thứ tự hai nhóm, cả hai khác rỗng) đạt hệ số tối ưu.

In [20]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def best_split(points):
    # TODO: tô màu G_d với d = đáp án
    pass

In [21]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_best_splits(points):
    # TODO: 2^(số thành phần liên thông của G_d - 1), xử lý trường hợp nhóm rỗng
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Khoảng cách 1, 1, 2. (2) Hai cạnh `1` tạo đường đi (hai phía); cạnh `2` nối hai đầu cùng màu ⇒ tam giác. (3) `2`.

**Bài 2.** (1) Khoảng cách 3, 3, 4, 4, 7, 7. (2) Cạnh 3 và 4 tạo chu trình 4 (chẵn). (3) Cạnh 7 (đường chéo) nối hai đỉnh cùng màu ⇒ `7`.

**Bài 3.** (1) Tính `d`. (2) BFS tô màu `G_d`. (3) Nếu một màu rỗng, đổi màu một đỉnh.

**Bài 4.** (1) Mỗi cặp ghét nhau là ràng buộc "khác màu". (2) Mâu thuẫn ⇒ `False`. (3) Hết ⇒ `True`.

**Bài 5.** (1) Mỗi thành phần liên thông của `G_d` có 2 cách tô. (2) `2^c` cách tô, chia 2 vì hoán đổi nhóm. (3) Trừ trường hợp một nhóm rỗng (chỉ xảy ra khi `G_d` không có cạnh).

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
assert Solution().maxPartitionFactor([[0, 0], [1, 0], [2, 0]]) == 2
assert Solution().maxPartitionFactor([[0, 0], [4, 0], [0, 3], [4, 3]]) == 7 == brute([[0, 0], [4, 0], [0, 3], [4, 3]])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [23]:
# Bài 3, 4
def best_split_sol(points):
    n = len(points); d = Solution().maxPartitionFactor(points)
    adj = [[j for j in range(n) if j != i and dist(points[i], points[j]) < d] for i in range(n)]
    color = [-1] * n
    for s in range(n):
        if color[s] == -1:
            color[s] = 0; q = deque([s])
            while q:
                u = q.popleft()
                for v in adj[u]:
                    if color[v] == -1: color[v] = color[u] ^ 1; q.append(v)
    if len(set(color)) == 1: color[0] ^= 1
    return color

def possible_bipartition_sol(n, dislikes):
    parent = list(range(n + 1)); par = [0] * (n + 1)
    def find(x):
        p = 0
        while parent[x] != x: p ^= par[x]; x = parent[x]
        return x, p
    for a, b in dislikes:
        (ra, pa), (rb, pb) = find(a), find(b)
        if ra == rb:
            if pa == pb: return False
        else: parent[ra] = rb; par[ra] = pa ^ pb ^ 1
    return True

rng = random.Random(4)
for _ in range(200):
    pts = [[rng.randint(-5, 5), rng.randint(-5, 5)] for _ in range(rng.randint(3, 8))]
    col = best_split_sol(pts); n = len(pts)
    f = min(dist(pts[i], pts[j]) for i, j in combinations(range(n), 2) if col[i] == col[j])
    assert f == brute(pts) and 0 < sum(col) < n
assert possible_bipartition_sol(4, [[1, 2], [1, 3], [2, 4]]) and not possible_bipartition_sol(3, [[1, 2], [1, 3], [2, 3]])
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [24]:
# Bài 5
def count_best_splits_sol(points):
    n = len(points); d = Solution().maxPartitionFactor(points)
    adj = [[j for j in range(n) if j != i and dist(points[i], points[j]) < d] for i in range(n)]
    seen = [False] * n; comps = 0; has_edge = any(adj)
    for s in range(n):
        if not seen[s]:
            comps += 1; seen[s] = True; q = deque([s])
            while q:
                u = q.popleft()
                for v in adj[u]:
                    if not seen[v]: seen[v] = True; q.append(v)
    ways = 2 ** comps // 2
    return ways - (0 if has_edge else 1)                    # bỏ cách "mọi điểm một nhóm"

def count_brute(points):
    n = len(points); best = brute(points); c = 0
    for mask in range(1, (1 << n) - 1):
        gs = [[i for i in range(n) if mask >> i & 1], [i for i in range(n) if not mask >> i & 1]]
        f = min((dist(points[i], points[j]) for g in gs for i, j in combinations(g, 2)), default=0)
        c += f == best
    return c // 2

rng = random.Random(5)
for _ in range(200):
    pts = [[rng.randint(-5, 5), rng.randint(-5, 5)] for _ in range(rng.randint(3, 8))]
    assert count_best_splits_sol(pts) == count_brute(pts), pts
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
if n == 2: return 0
edges = sorted((dist(i, j), i, j) for mọi cặp)
for w, i, j in edges:                 # DSU chẵn lẻ: i, j phải khác màu
    (ri, pi), (rj, pj) = find(i), find(j)
    if ri == rj and pi == pj: return w   # chu trình lẻ đầu tiên
    if ri != rj: parent[ri] = rj; par[ri] = pi ^ pj ^ 1
```

**Checklist:**

- [ ] `n = 2` ⇒ `0`?
- [ ] Cạnh là cặp có khoảng cách **`< d`** (nghiêm ngặt)?
- [ ] DSU nén đường có cập nhật chẵn lẻ?
- [ ] Duyệt cạnh theo thứ tự **tăng dần**?
- [ ] Đã so với thử mọi cách chia?